# Wheel Profile Shape and Empty-Car Hunting

I test whether tread-shape measurements add information beyond mean conicity
and rail profile. This is one fixed feature comparison using the same Gaussian
kernel ridge classifier and car-grouped validation as the contact experiment.


In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd

project_folder = Path.cwd()
if project_folder.name == "notebooks":
    project_folder = project_folder.parent
sys.path.insert(0, str(project_folder / "src"))

from contact_hunting import (
    load_tables, prepare_trucks, evaluate_models,
    paired_car_intervals, classification_metrics,
)
from profile_shapes import prepare_truck_shapes, SHAPE_MODELS, SHAPE_COMPARISONS

data_folder = Path(os.environ.get("HUNTING_DATA_DIR", project_folder / "data"))
merged_data, contact_data = load_tables(data_folder)
truck_data = prepare_trucks(merged_data, contact_data)
shape_data, wheel_data, profiles = prepare_truck_shapes(
    data_folder / "Profiles" / "Candidate wheel profiles", truck_data
)
truck_data = truck_data.merge(shape_data, on=["Car", "Truck"], validate="many_to_one")
display(pd.Series({
    "Wheel profiles": len(wheel_data),
    "Cars": truck_data.Car.nunique(),
    "Trucks": len(shape_data),
    "Truck/rail assessments": len(truck_data),
}, name="Coverage"))


## Shape measurements

The reader checks point counts, finite coordinates, increasing x positions in
the tread band, and coverage of x=60–110. It interpolates at one-unit spacing
and subtracts the height at x=85. All processing happens in memory.

Three measurements describe each truck:

- Mean fitted tread slope across its four wheels.
- Mean RMS departure from a straight line across its four wheels.
- Larger left/right RMS shape difference across its two wheelsets.

The window and measurements were fixed before this comparison. Both sides use
the export's existing x orientation. Units, alignment, and reconstruction
provenance remain provisional; these are exported coordinate descriptors,
not validated physical curvature or equivalent conicity.


## Evaluation

One complete car is held out at a time. Three grouped folds within the training
cars choose regularization and kernel width. Scaling is fitted on training
rows only. Both feature sets use the same six-setting grid and distance scale.

Acceleration is the main exploratory criterion; displacement is a separate
sensitivity check. A speed below 80 mph indicates hunting. For displacement,
the lower of the two axle speeds determines the truck label. A value of 80
means no hunting recorded within the tested range.


In [ ]:
target_names = {
    "acceleration_hunted": "Acceleration",
    "displacement_hunted": "Displacement sensitivity",
}
metric_tables = []
comparison_tables = []
local_predictions = {}

for target, label in target_names.items():
    metrics, predictions, settings = evaluate_models(truck_data, target, SHAPE_MODELS)
    intervals, comparisons = paired_car_intervals(
        predictions, n_bootstrap=2000, seed=42, comparisons=SHAPE_COMPARISONS
    )
    metrics["criterion"] = label
    comparisons["criterion"] = label
    metric_tables.append(metrics)
    comparison_tables.append(comparisons)
    local_predictions[target] = predictions

results = pd.concat(metric_tables, ignore_index=True)
comparisons = pd.concat(comparison_tables, ignore_index=True)
performance = results.set_index(["criterion", "model"])
display((100 * performance[["balanced_accuracy", "recall", "specificity"]]).round(1)
        .rename(columns={"balanced_accuracy": "Balanced accuracy (%)",
                         "recall": "Hunting recall (%)",
                         "specificity": "No-hunting recall (%)"}))


## Incremental improvement

Balanced accuracy averages hunting recall and no-hunting recall. The paired
ranges resample complete cars 2,000 times from fixed held-out predictions.
They do not include retraining or uncertainty from exploratory feature choices.
A range crossing zero indicates an inconclusive comparison.


In [ ]:
paired = comparisons.set_index("criterion")
comparison_table = (100 * paired[
    ["change_in_balanced_accuracy", "lower_95", "upper_95"]
]).round(1).rename(columns={
    "change_in_balanced_accuracy": "Gain (percentage points)",
    "lower_95": "95% lower", "upper_95": "95% upper",
})
comparison_table["Cars with fewer errors"] = paired.cars_with_fewer_errors
comparison_table["Cars with more errors"] = paired.cars_with_more_errors
comparison_table["Cars tied"] = paired.cars_tied
display(comparison_table)


## Sensitivity to individual cars

I recalculate the score difference after omitting each car's held-out predictions
in turn. This checks whether one car accounts for the gain. It does not refit
models without that car and is not a second validation experiment.


In [ ]:
influence_rows = []
for target, predictions in local_predictions.items():
    for car in sorted(predictions.Car.unique()):
        remaining = predictions.loc[predictions.Car != car]
        base = classification_metrics(remaining.actual, remaining["Mean conicity + rail"])
        shape = classification_metrics(remaining.actual, remaining["Mean conicity + shape + rail"])
        influence_rows.append({
            "criterion": target_names[target], "omitted_car": car,
            "gain": shape["balanced_accuracy"] - base["balanced_accuracy"],
        })

influence = pd.DataFrame(influence_rows)
display((100 * influence.groupby("criterion").gain.agg(["min", "max"]))
        .round(1).rename(columns={"min": "Smallest gain (pp)",
                                 "max": "Largest gain (pp)"}))


## Interpretation

The comparison tests the three descriptors together. A gain does not identify
which measurement explains hunting or establish a cause. These features may
also reflect other differences between cars.

Whole-car splits prevent the same car's wheels and rail assessments from
appearing in training and test together. The number of independent cars remains
small, and repeated exploration of this dataset makes these results provisional.
The displacement comparison stays separate from the main acceleration result.
The experiment does not test new rail types or operating conditions.
